In [0]:
#####################
#Define This notebook name
this_notebook_state = "paymentPending"
# this_notebook_state = "appealSubmitted"
# this_notebook_state = "awaitingRespondentEvidence(a)"
# this_notebook_state = "awaitingRespondentEvidence(b)"
# this_notebook_state = "caseUnderReview"
# this_notebook_state = "reasonsForAppealSubmitted"
# this_notebook_state = "listing"
# this_notebook_state = "prepareForHearing"
# this_notebook_state = "decision"
# this_notebook_state = "decided(a)"
# this_notebook_state = "decided(b)"
# this_notebook_state = "ftpaSubmitted"
# this_notebook_state = "ftpaDecided"
# this_notebook_state = "remitted"

In [0]:
########################
#Setup for Tranformation Tests
########################

#Setup Test Reporting
from datetime import datetime, timedelta
run_user = spark.sql("SELECT current_user()").first()[0]
run_tag = "Testing Transformation Tests"
run_by_automation_name = "Transformation_Tests"
#Capture Test Start datetime
run_start_datetime = datetime.now()
#####################

#Setup Notebook Parameters (Defaulting to Payment Pending and Running all tests)
dbutils.widgets.text("state_under_test", "")

#fields_to_exclude : Should be a comma separated list of fields to exclude
dbutils.widgets.text("fields_to_exclude", "")

# Read parameters
state_under_test = dbutils.widgets.get("state_under_test")
fields_to_exclude = dbutils.widgets.get("fields_to_exclude")

#Set Default Values if not called from another Notebook
if state_under_test == "" :
    state_under_test = this_notebook_state

#Get Fields to Exclude
if fields_to_exclude != "":
    fields_to_exclude = [item.strip() for item in fields_to_exclude.split(",")]
else:
    fields_to_exclude = []

#List of Fields to Exclude in Child Notebooks Called
child_fields_to_exclude = []

#Combine any Fields to Exclude
fields_to_exclude.extend(child_fields_to_exclude)

print(f"Fields to exclude = {str(fields_to_exclude)}")
print(f"Testing State = {state_under_test}")

#Restart Python When needed (when changed tests)
# dbutils.library.restartPython()

#Import Tests

#Import models.test_result as test_result
from models.test_result import TestResult

#Import asdict to convert Dataclass to Dictionary
from dataclasses import asdict
import os

#Setup Global Variables
all_test_results = []
current_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
base_path = current_path.rsplit("/", 1)[0] + "/"
#Below will be replaced eventually to store the reults in a spark table
test_results_path= "/Workspace/Users/" + run_user + "/Results/Transformation_Tests"
os.makedirs(test_results_path, exist_ok=True)


In [0]:
#Load Config and Setup Enviorment Variables
M2_bronze = None
M3_bronze = None
M3_silver = None
M6_bronze = None
bat = None
bhoref = None
bac = None
bll = None
b = None
M4_silver = None
M4_bronze = None
docsr = None
H_bronze = None
H_silver = None

config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()

KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"

# Service principal credentials
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")

# Storage account names
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"
external_storage = f"ingest{lz_key}external{env_name}"

# Spark config for curated storage
spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{checkpoint_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{checkpoint_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{checkpoint_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{checkpoint_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{checkpoint_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for raw storage
spark.conf.set(f"fs.azure.account.auth.type.{raw_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{raw_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{raw_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{raw_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{raw_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for landing storage
spark.conf.set(f"fs.azure.account.auth.type.{landing_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{landing_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{landing_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{landing_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{landing_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for external storage
spark.conf.set(f"fs.azure.account.auth.type.{external_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{external_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{external_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{external_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{external_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Setting variables for use in subsequent cells
bronze_path = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
silver_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
gold_path = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/{state_under_test}"

import json

#Get Latest Json Folder
json_location = dbutils.fs.ls(f"{gold_path}/")[-1]
latest_json_location = json_location.name

#Set Paths
try:
    json_path = f"{gold_path}/{latest_json_location}/JSON/"
    M1_silver = f"{silver_path}/silver_appealcase_detail"
    M1_bronze = f"{bronze_path}/bronze_appealcase_crep_rep_floc_cspon_cfs"
    M2_bronze = f"{bronze_path}/bronze_appealcase_caseappellant_appellant"
    M3_bronze = f"{bronze_path}/bronze_status_htype_clist_list_ltype_court_lsitting_adj"
    M2_silver = f"{silver_path}/silver_caseapplicant_detail"
    M3_silver = f"{silver_path}/silver_status_detail"
    C = f"{silver_path}/silver_appealcategory_detail"
    bhc = f"{bronze_path}/bronze_hearing_centres"
    bat = f"{bronze_path}/bronze_appealtype"
    docsr = f"{bronze_path}/bronze_documentsreceived"
    H_bronze = f"{bronze_path}/bronze_history"
    H_silver = f"{silver_path}/silver_history_detail"
    bhoref = f"{bronze_path}/bronze_HORef_cleansing"
except:
    print(f"Error during fetch: {str(e)}")

#Create and Load Dataframes
json_data = spark.read.format("json").load(json_path)
M1_silver = spark.read.format("delta").load(M1_silver)
M1_bronze = spark.read.format("delta").load(M1_bronze)
M2_bronze = spark.read.format("delta").load(M2_bronze)
M2_silver = spark.read.format("delta").load(M2_silver)
M3_silver = spark.read.format("delta").load(M3_silver)
M3_bronze = spark.read.format("delta").load(M3_bronze)
C = spark.read.format("delta").load(C)
bhc = spark.read.format("delta").load(bhc)
bat = spark.read.format("delta").load(bat)
docsr = spark.read.format("delta").load(docsr)
H_bronze = spark.read.format("delta").load(H_bronze)
H_silver = spark.read.format("delta").load(H_silver)
bhoref = spark.read.format("delta").load(bhoref)

# Load additional tables (may not exist in all environments)
try: M4_silver = spark.read.format("delta").load(f"{silver_path}/silver_transaction_detail") if M4_silver is None else M4_silver
except: pass
try: M4_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_appealcase_transaction_transactiontype") if M4_bronze is None else M4_bronze
except: pass
try: M6_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_history") if M6_bronze is None else M6_bronze
except: pass
try: bac = spark.read.format("delta").load(f"{bronze_path}/bronze_appealcategory") if bac is None else bac
except: pass
try: bll = spark.read.format("delta").load(f"{bronze_path}/bronze_listing_location") if bll is None else bll
except: pass

from pyspark.sql.functions import (
    col, when, lit, array, struct, collect_list,
    max as spark_max, date_format, row_number, expr,
    size, udf, coalesce, concat_ws, concat, trim, year, split, datediff,
    collect_set, current_timestamp,transform, first, array_contains
)

In [0]:
from pyspark.sql import functions as F
import re

test_df = json_data.join(M1_bronze, M1_bronze["CaseNo"] == json_data["appealReferenceNumber"], "left").drop("CaseNo")
test_df = test_df.select("appealReferenceNumber", "TTL", "DateLodged")

print(f"State under test: {this_notebook_state}")
test_df.display()

In [0]:
# TTL date fails
TTL_date_fails = test_df.filter(
    col("TTL.SystemTTL").cast("date") != F.add_months(col("DateLodged"), 1200).cast("date")
)

if TTL_date_fails.count() != 0:
    all_test_results.append("TTL date fails: there are cases where TTL.SystemTTL does not equal DateLodged + 100 years/1200 months")
    print("Test fails:")
    TTL_date_fails.display()
else:
    all_test_results.append("TTL date passes: all cases have TTL.SystemTTL which equal DateLodged + 100 years/1200 months")

print("Displaying SystemTTL with M1.DateLodged + 100 years below")
test_df.select("appealReferenceNumber", 
               (col("TTL.SystemTTL").cast("date")).alias("SystemTTL"), 
               (F.add_months(col("DateLodged"), 1200).cast("date")).alias("DateLodged")
            ).display()

In [0]:
# TTL suspended fails
TTL_suspended_fails = test_df.filter(
    col("TTL.Suspended") != "No"
)

if TTL_suspended_fails.count() != 0:
    all_test_results.append("TTL suspended fails: there are cases where TTL.Suspended does not equal No")
    print("Test fails:")
    TTL_suspended_fails.display()
else:
    all_test_results.append("TTL suspended passes: all cases have TTL.Suspended equal to No")

print("Displaying Suspended below")
test_df.select("appealReferenceNumber", 
               "TTL.Suspended"
            ).display()

In [0]:
# TTL date fails
# ISO‑8601 format e.g. “2126-01-14”
TTL_dateFormat_fails =test_df.filter(
    (~col("TTL.SystemTTL").rlike(r"^\d{4}-\d{2}-\d{2}$")) |
    (col("TTL.SystemTTL").cast("date").isNull())
)

if TTL_dateFormat_fails.count() != 0:
    all_test_results.append("TTL date format fails: there are cases where TTL.SystemTTL does not follow the correct ISO-8601 format")
    print("Test fails:")
    TTL_dateFormat_fails.display()
else:
    all_test_results.append("TTL date format passes: all cases have a TTL.SystemTTL value which follows the correct ISO-8601 format")

# TTL fails

print("Displaying SystemTTL below")
test_df.select("appealReferenceNumber", 
               "TTL.SystemTTL"
            ).display()

In [0]:
print(f"Results for {this_notebook_state}")
for result in all_test_results:
    print(result)